# Подготовка

In [ ]:
import os
# from pathlib import Path
# import shutil
# import json
import requests
# from urllib.parse import quote
import re

# Константы
MYINDIE_JAM_URL = "https://myindie.ru/jams/jam/myindie-level-10/games?page="
MYINDIE_JAM_URL_PAGES = 5  # Количество страниц с играми на геймджеме
OUTPUT_DIR = "output/"

In [12]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. Скачать все страницы геймджема

Найти все игры на джеме

In [ ]:
def get_jam_games_urls(jam_url):
    """Scrapes the jam page and extracts game URLs."""

    response = requests.get(jam_url)

    if response.status_code != 200:
        print(f"Error fetching jam page: `{response.status_code}`")
        return []

    games_urls = re.findall(r'/games/game/[\w-]+', response.text)
    games_urls = [f"https://myindie.ru{url}" for url in games_urls]
    print(f"Found {len(games_urls)} games URLs on: {jam_url}")

    return games_urls


games_urls = []
for page in range(1, MYINDIE_JAM_URL_PAGES + 1):
    paged_url = f"{MYINDIE_JAM_URL}{page}"
    games_urls.extend(get_jam_games_urls(paged_url))

print(f"\n{len(games_urls)} total games found across {MYINDIE_JAM_URL_PAGES} pages:\n{chr(10).join(games_urls)}")


Found 30 game URLs: ['https://myindie.ru/games/game/my-tedious-routine', 'https://myindie.ru/games/game/manlygun', 'https://myindie.ru/games/game/koronka-zhmyot', 'https://myindie.ru/games/game/rassudok', 'https://myindie.ru/games/game/sedmoj', 'https://myindie.ru/games/game/burn-them-all', 'https://myindie.ru/games/game/kill-switch_wov', 'https://myindie.ru/games/game/clone-ban', 'https://myindie.ru/games/game/remember-you-are-loved', 'https://myindie.ru/games/game/beshenyj-byk_wvp', 'https://myindie.ru/games/game/dont-peer-into-the-dark', 'https://myindie.ru/games/game/povinujsya', 'https://myindie.ru/games/game/frosty-trouble', 'https://myindie.ru/games/game/diagnoz', 'https://myindie.ru/games/game/proklyataya-ten_a0i', 'https://myindie.ru/games/game/vsyo-kak-vsegda', 'https://myindie.ru/games/game/kotolaba', 'https://myindie.ru/games/game/music-factory', 'https://myindie.ru/games/game/miholk-vdali-ot-tela', 'https://myindie.ru/games/game/brazen-meat', 'https://myindie.ru/games/game

Скачать все HTML-страницы игр геймджема

In [17]:
jam_games_file_path = os.path.join(OUTPUT_DIR, f"jam_games.txt")
if os.path.exists(jam_games_file_path):
    os.remove(jam_games_file_path)
jam_games_file = open(jam_games_file_path, 'a', encoding='utf-8')

DEBUG_GAMES_MAX = 2  # DEBUG delete
for i, game_url in enumerate(games_urls[:DEBUG_GAMES_MAX]):
    print(f"Processing game URL: `{game_url}`")
    response = requests.get(game_url)
    if response.status_code != 200:
        print(f"* Error fetching game page: `{response.status_code}`")
        continue

    title_match = re.search(r'<title>([^<]+)</title>', response.text)
    if title_match:
        title = f"{i:03d}_" + re.sub(r'[^\w_]', '-', re.sub(r'\s+', '_', title_match.group(1).strip()))
    else:
        title = f"{i:03d}_" + "Unknown"

    output_file_path = os.path.join(OUTPUT_DIR, f"{title}.html")
    with open(output_file_path, 'w', encoding='utf-8') as f:
        f.write(response.text)
        print(f"Saved game `{title_match}` to `{output_file_path}`")
    jam_games_file.write(f"{output_file_path} {game_url}\n")

jam_games_file.close()

Processing game URL: `https://myindie.ru/games/game/my-tedious-routine`
Saved game `<re.Match object; span=(194, 271), match='<title>My Tedious Routine. Жанр: Arcade, Action |>` to `output/000_My_Tedious_Routine-_Жанр-_Arcade-_Action_-_Инди-игры_-_MyIndie.html`
Processing game URL: `https://myindie.ru/games/game/manlygun`
Saved game `<re.Match object; span=(194, 254), match='<title>ManlyGun. Жанр: Shooter | Инди-игры | MyIn>` to `output/001_ManlyGun-_Жанр-_Shooter_-_Инди-игры_-_MyIndie.html`
